In [2]:
!pip install torch torchvision rdkit transformers gradio -qqq

In [3]:
import gradio as gr
from rdkit import Chem
from rdkit.Chem import Draw
from transformers import AutoTokenizer, EncoderDecoderModel


pretrained_model = "ribesstefano/ChemBERTa2ChemBERTa-58M"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
bert2bert = EncoderDecoderModel.from_pretrained(pretrained_model)


def get_mol_img(smiles):
    return Chem.Draw.MolToImage(Chem.MolFromSmiles(smiles))


def split_protac(protac_smiles):
    protac_smiles_tokenized = tokenizer(protac_smiles, return_tensors="pt")
    predictions = bert2bert.generate(
        input_ids=protac_smiles_tokenized["input_ids"],
        attention_mask=protac_smiles_tokenized["attention_mask"],
    )
    split_protacs = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True).replace(" ", "") for g in predictions]
    split_protac = split_protacs[0]
    # Get the images input and output molecules
    protac_smiles_img = get_mol_img(protac_smiles)
    split_protac_img = get_mol_img(split_protac)
    # Get the images of the substructures
    substructures = split_protac.split('.')
    substructures_img = []
    for substructure in substructures:
        substructures_img.append(get_mol_img(substructure))
    # Add None images if the split didn't produce three substructures
    for i in range(3 - len(substructures_img)):
        substructures_img.append(None)
    return split_protac, protac_smiles_img, split_protac_img, *(tuple(substructures_img))


default_protac = "NC(=O)CC[C@H](NC(=O)[C@@H]1CC[C@@H]2CCN(C(=O)CCCCCC#Cc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)C[C@H](NC(=O)c3cc4cc(C(F)(F)P(=O)(O)O)ccc4[nH]3)C(=O)N12)C(=O)NCc1ccccc1"

demo = gr.Interface(
    fn=split_protac,
    inputs=gr.Textbox(lines=1, value=default_protac, label="PROTAC SMILES to split"),
    outputs=[
        gr.Textbox(lines=1, label="Split PROTAC SMILES"),
        gr.Image(type="pil", label="Input PROTAC"),
        gr.Image(type="pil", label="Split PROTAC"),
        gr.Image(type="pil", label="PROTAC sub-structure n.1"),
        gr.Image(type="pil", label="PROTAC sub-structure n.2"),
        gr.Image(type="pil", label="PROTAC sub-structure n.3"),
    ],
)

demo.launch()

The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.

To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>